# Distribution of diseases across therapeutic areas

**R3-mn-4** — *"The manuscript does not appear to include a dedicated section describing how traits are
distributed across therapeutic areas. A summary of disease or trait distribution across therapeutic areas
would be useful."*

This notebook builds that table: **how many diseases fall in each therapeutic area**, over the two trait
universes the manuscript uses.

| universe | what it is | size |
| -------- | ---------- | ---- |
| **qualifying dataset** | ontology terms used by studies that passed QC and entered the analyses | 2,320 disease terms (+ 7,010 measurement terms) |
| **gPS disease list** | terms with ≥ 1 qualifying credible set carrying an L2G-prioritised gene — the universe gPS and gps_TA are computed over | 1,394 terms |

The genetic-correlation half of R3-mn-4 is answered separately in `../ta-independence/`.

## One area per disease, by the manuscript's own rule

`therapy_area_hierarchy` from `chapters/01-data-preparation/04_qualifying_dataset_generation.ipynb`: the
first root found in a term's `ancestors` list, in the dictionary's own order; `other` if none. This is
reproduced here exactly rather than reimplemented — the check below confirms it regenerates the stored
`mappedTherapeuticAreas` for all 15,730 qualifying studies, so the counts are on the paper's convention,
not a variant of it.

In [1]:
import ast
import itertools
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 120)

INTERMEDIATE = "../../../data/intermediate_files/"
RELEASE = "../../../data/25.06/"

In [2]:
# Verbatim from 04_qualifying_dataset_generation.ipynb. Dict order IS the priority order.
THERAPY_AREA_HIERARCHY = {
    "EFO_0001444": "measurement",
    "MONDO_0045024": "cancer or benign tumor",
    "EFO_0005741": "infectious disease",
    "OTAR_0000009": "injury, poisoning or other complication",
    "OTAR_0000014": "pregnancy or perinatal disease",
    "MONDO_0024458": "disorder of visual system",
    "EFO_0000319": "cardiovascular disease",
    "EFO_0009605": "pancreas disease",
    "EFO_0000540": "immune system disease",
    "EFO_0010282": "gastrointestinal disease",
    "OTAR_0000017": "reproductive system or breast disease",
    "EFO_0010285": "integumentary system disease",
    "EFO_0001379": "endocrine system disease",
    "OTAR_0000010": "respiratory or thoracic disease",
    "EFO_0009690": "urinary system disease",
    "OTAR_0000006": "musculoskeletal or connective tissue disease",
    "MONDO_0021205": "disorder of ear",
    "EFO_0005803": "hematologic disease",
    "EFO_0000618": "nervous system disease",
    "MONDO_0002025": "psychiatric disorder",
    "OTAR_0000020": "nutritional or metabolic disease",
    "OTAR_0000018": "genetic, familial or congenital disease",
    "EFO_0003765": "sign or symptom",
}

onto = pd.read_parquet(RELEASE + "output/disease/disease.parquet", columns=["id", "name", "ancestors"])
NAME = onto.set_index("id")["name"]
ANCESTORS = {r.id: set(r.ancestors) if r.ancestors is not None else set() for r in onto.itertuples()}
IN_ONTOLOGY = set(onto["id"])


def area_of(term):
    """The pipeline's rule, unchanged: first hierarchy root present in the term's ancestors, else `other`.

    Note this tests `ancestors` only, and a term is not its own ancestor -- so a term that IS an area root
    is labelled `other`. That is the upstream behaviour and is reproduced deliberately; the 13 terms it
    affects are listed at the end.
    """
    ancestors = ANCESTORS.get(term, set())
    for root in THERAPY_AREA_HIERARCHY:
        if root in ancestors:
            return root
    return "other"


print("ontology terms:", len(IN_ONTOLOGY))

ontology terms: 38959


### Proof that this is the pipeline's rule and not a lookalike

The term → area map is not persisted anywhere — `04_qualifying_dataset_generation.ipynb` builds it inside a
Spark UDF closure and only the *study-level* result is written out. So it has to be rebuilt here. The check
is that rebuilding it and re-deriving each study's `mappedTherapeuticAreas` reproduces the stored column
exactly.

In [3]:
qualifying = pd.read_parquet(
    INTERMEDIATE + "qualifying_gwas_studies", columns=["studyId", "diseaseIds", "mappedTherapeuticAreas"]
)
matches = sum(
    {area_of(t) for t in row.diseaseIds} == set(row.mappedTherapeuticAreas) for row in qualifying.itertuples()
)
print(f"studies whose stored mappedTherapeuticAreas is reproduced: {matches} of {len(qualifying)}")
assert matches == len(qualifying), "the rebuilt map is not the pipeline's map"

studies whose stored mappedTherapeuticAreas is reproduced: 15730 of 15730


## The two universes

In [4]:
QUALIFYING_DISEASE = set(itertools.chain.from_iterable(qualifying["diseaseIds"].map(list)))

measurement_studies = pd.read_parquet(
    INTERMEDIATE + "qualifying_measurement_studies", columns=["studyId", "diseaseIds"]
)
QUALIFYING_MEASUREMENT = set(itertools.chain.from_iterable(measurement_studies["diseaseIds"].map(list)))

genes_df = pd.read_csv(INTERMEDIATE + "genes_therapeutic_areas.csv")
l2g = pd.read_csv(INTERMEDIATE + "l2g_diseases_full-r1.csv", usecols=["geneId", "diseaseIds"])
l2g = l2g[l2g["geneId"].isin(set(genes_df["geneId"]))]
GPS_DISEASE = set(itertools.chain.from_iterable(l2g["diseaseIds"].map(ast.literal_eval)))

print("qualifying disease terms:  ", len(QUALIFYING_DISEASE))
print("qualifying measurement terms:", len(QUALIFYING_MEASUREMENT))
print("gPS disease-list terms:    ", len(GPS_DISEASE))
assert len(QUALIFYING_DISEASE) == 2320 and len(GPS_DISEASE) == 1394
assert GPS_DISEASE <= QUALIFYING_DISEASE, "the gPS list must be a subset of the qualifying terms"

qualifying disease terms:   2320
qualifying measurement terms: 7010
gPS disease-list terms:     1394


## The table

In [5]:
qual_counts = Counter(area_of(t) for t in QUALIFYING_DISEASE)
gps_counts = Counter(area_of(t) for t in GPS_DISEASE)

rows = []
for root, label in THERAPY_AREA_HIERARCHY.items():
    if root == "EFO_0001444":
        rows.append(
            {
                "therapeutic_area": "measurement",
                "root_id": root,
                "trait_class": "measurement",
                "n_diseases_qualifying": len(QUALIFYING_MEASUREMENT),
                "n_diseases_gps": np.nan,
            }
        )
        continue
    rows.append(
        {
            "therapeutic_area": label,
            "root_id": root,
            "trait_class": "disease",
            "n_diseases_qualifying": qual_counts.get(root, 0),
            "n_diseases_gps": gps_counts.get(root, 0),
        }
    )
rows.append(
    {
        "therapeutic_area": "other (no area root)",
        "root_id": "",
        "trait_class": "disease",
        "n_diseases_qualifying": qual_counts.get("other", 0),
        "n_diseases_gps": gps_counts.get("other", 0),
    }
)

table = pd.DataFrame(rows)
disease_rows = table["trait_class"] == "disease"

# the rule assigns exactly one area per term, so the disease rows must sum to the universe
assert table.loc[disease_rows, "n_diseases_qualifying"].sum() == len(QUALIFYING_DISEASE)
assert table.loc[disease_rows, "n_diseases_gps"].sum() == len(GPS_DISEASE)

table["pct_qualifying"] = (100 * table["n_diseases_qualifying"] / len(QUALIFYING_DISEASE)).round(2)
table["pct_gps"] = (100 * table["n_diseases_gps"] / len(GPS_DISEASE)).round(2)
table.loc[~disease_rows, ["pct_qualifying", "pct_gps"]] = np.nan

table = pd.concat(
    [
        table,
        pd.DataFrame(
            [
                {
                    "therapeutic_area": "TOTAL (disease side)",
                    "root_id": "",
                    "trait_class": "disease",
                    "n_diseases_qualifying": len(QUALIFYING_DISEASE),
                    "n_diseases_gps": len(GPS_DISEASE),
                    "pct_qualifying": 100.0,
                    "pct_gps": 100.0,
                }
            ]
        ),
    ],
    ignore_index=True,
)

table.to_csv(INTERMEDIATE + "ta_distribution_supplementary-r1.csv", index=False)
print(table.to_string(index=False))

                            therapeutic_area       root_id trait_class  n_diseases_qualifying  n_diseases_gps  pct_qualifying  pct_gps
                                 measurement   EFO_0001444 measurement                   7010             NaN             NaN      NaN
                      cancer or benign tumor MONDO_0045024     disease                    350           240.0           15.09    17.22
                          infectious disease   EFO_0005741     disease                    142            53.0            6.12     3.80
     injury, poisoning or other complication  OTAR_0000009     disease                     65            39.0            2.80     2.80
              pregnancy or perinatal disease  OTAR_0000014     disease                     18            11.0            0.78     0.79
                   disorder of visual system MONDO_0024458     disease                    117            85.0            5.04     6.10
                      cardiovascular disease   EFO_0000

### Sorted by size

In [6]:
view = table[(table["trait_class"] == "disease") & (table["therapeutic_area"] != "TOTAL (disease side)")].copy()
view = view.sort_values("n_diseases_qualifying", ascending=False)
print(
    view[["therapeutic_area", "n_diseases_qualifying", "pct_qualifying", "n_diseases_gps", "pct_gps"]].to_string(
        index=False
    )
)

named = view[view["root_id"] != ""]
print(f"\nareas with >=1 qualifying disease: {int((named['n_diseases_qualifying'] > 0).sum())} of 22")
print(f"areas with >=1 gPS-list disease:   {int((named['n_diseases_gps'] > 0).sum())} of 22")
print(
    f"share of qualifying diseases in `other`: {float(view.loc[view['root_id'] == '', 'pct_qualifying'].iloc[0]):.1f}%"
)
print(f"share of gPS diseases in `other`:        {float(view.loc[view['root_id'] == '', 'pct_gps'].iloc[0]):.1f}%")

                            therapeutic_area  n_diseases_qualifying  pct_qualifying  n_diseases_gps  pct_gps
                        other (no area root)                    586           25.26           303.0    21.74
                      cancer or benign tumor                    350           15.09           240.0    17.22
                      nervous system disease                    175            7.54            90.0     6.46
                      cardiovascular disease                    161            6.94           116.0     8.32
                          infectious disease                    142            6.12            53.0     3.80
                   disorder of visual system                    117            5.04            85.0     6.10
                       immune system disease                    111            4.78            75.0     5.38
musculoskeletal or connective tissue disease                     97            4.18            66.0     4.73
                   

## How uneven is the distribution

Over the 22 named areas only, since `other` is a residual bucket rather than a therapeutic area.

In [7]:
def concentration(counts):
    counts = np.asarray([c for c in counts if c > 0], dtype=float)
    share = counts / counts.sum()
    gini = float(np.abs(np.subtract.outer(share, share)).sum() / (2 * len(share)))
    return {
        "areas_occupied": int(len(counts)),
        "min": int(counts.min()),
        "median": float(np.median(counts)),
        "max": int(counts.max()),
        "largest_share_pct": round(100 * share.max(), 2),
        "top3_share_pct": round(100 * np.sort(share)[-3:].sum(), 2),
        "gini": round(gini, 3),
        "shannon_evenness": round(float(-(share * np.log(share)).sum() / np.log(len(share))), 3),
    }


spread = pd.DataFrame(
    [
        dict(universe="qualifying disease terms", **concentration(named["n_diseases_qualifying"])),
        dict(universe="gPS disease-list terms", **concentration(named["n_diseases_gps"])),
    ]
)
spread.to_csv(INTERMEDIATE + "ta_distribution_concentration-r1.csv", index=False)
print(spread.to_string(index=False))

                universe  areas_occupied  min  median  max  largest_share_pct  top3_share_pct  gini  shannon_evenness
qualifying disease terms              22    8    52.5  350              20.18           39.56 0.478             0.875
  gPS disease-list terms              22    2    34.5  240              22.00           40.88 0.479             0.870


## Term-level backing table

One row per disease term with its area, so any cell above can be checked.

In [8]:
term_rows = []
for term in sorted(QUALIFYING_DISEASE):
    root = area_of(term)
    term_rows.append(
        {
            "term": term,
            "name": NAME.get(term, ""),
            "therapeutic_area": THERAPY_AREA_HIERARCHY.get(root, "other (no area root)"),
            "in_gps_disease_list": term in GPS_DISEASE,
        }
    )
terms = pd.DataFrame(term_rows).sort_values(["therapeutic_area", "name"])
terms.to_csv(INTERMEDIATE + "ta_distribution_terms-r1.csv", index=False)
print("term-level rows:", len(terms))
print(terms.head(10).to_string(index=False))

term-level rows: 2320
         term                                                                              name       therapeutic_area  in_gps_disease_list
  EFO_0000094                                               B-cell acute lymphoblastic leukemia cancer or benign tumor                 True
MONDO_0600030 B-cell acute lymphoblastic leukemia with t(1;19)(q23;p13.3); E2A-PBX1 (TCF3-PBX1) cancer or benign tumor                False
  EFO_0009443                                                               BRCAX breast cancer cancer or benign tumor                 True
  EFO_1000107                                                             Benign Brain Neoplasm cancer or benign tumor                 True
  EFO_1000110                                                      Benign Conjunctival Neoplasm cancer or benign tumor                False
  EFO_1000111                                                            Benign Kidney Neoplasm cancer or benign tumor                Fals

## Why a quarter of diseases have no therapeutic area

`other` is the second-largest row, so it needs an explanation rather than a footnote.

In [9]:
unmapped = terms[terms["therapeutic_area"] == "other (no area root)"].copy()
unmapped["prefix"] = unmapped["term"].str.split("_").str[0]
unmapped["in_ontology_index"] = unmapped["term"].isin(IN_ONTOLOGY)
breakdown = pd.crosstab(unmapped["prefix"], unmapped["in_ontology_index"], margins=True)
breakdown.to_csv(INTERMEDIATE + "ta_distribution_other_breakdown-r1.csv")
print(f"qualifying disease terms with no therapeutic area: {len(unmapped)}")
print("by ontology prefix (columns: present in the disease index?)")
print(breakdown.to_string())

roots_used = [t for t in QUALIFYING_DISEASE if t in THERAPY_AREA_HIERARCHY]
print(f"\nof these, terms that ARE a therapeutic-area root: {len(roots_used)}")
print("the upstream rule tests `ancestors` only and a term is not its own ancestor, so a GWAS annotated")
print("directly to an area root is labelled `other`. Reproduced as-is; listed here for disclosure:")
for t in sorted(roots_used, key=lambda x: THERAPY_AREA_HIERARCHY[x]):
    print(f"   {t:15s} {THERAPY_AREA_HIERARCHY[t]}")

qualifying disease terms with no therapeutic area: 586
by ontology prefix (columns: present in the disease index?)
in_ontology_index  True  All
prefix                      
EFO                 166  166
GO                   24   24
HP                  377  377
MONDO                 3    3
MP                    1    1
OBA                  15   15
All                 586  586

of these, terms that ARE a therapeutic-area root: 13
the upstream rule tests `ancestors` only and a term is not its own ancestor, so a GWAS annotated
directly to an area root is labelled `other`. Reproduced as-is; listed here for disclosure:
   EFO_0000319     cardiovascular disease
   MONDO_0021205   disorder of ear
   MONDO_0024458   disorder of visual system
   EFO_0001379     endocrine system disease
   EFO_0010282     gastrointestinal disease
   EFO_0005803     hematologic disease
   EFO_0000540     immune system disease
   EFO_0005741     infectious disease
   EFO_0000618     nervous system disease
   EFO_0009

## Exports

| File | Contents |
| ---- | -------- |
| `ta_distribution_supplementary-r1.csv` | **the table** — diseases per therapeutic area and percentage, for the qualifying dataset and the gPS disease list |
| `ta_distribution_terms-r1.csv` | one row per disease term with its area and gPS-list membership |
| `ta_distribution_concentration-r1.csv` | how uneven the distribution is (Gini, evenness, top-3 share) |
| `ta_distribution_other_breakdown-r1.csv` | the unmapped terms by ontology prefix |